# 2장 1강: AARRR 프레임워크 개요와 단계별 분석 목적 — 실습문제

## 실습 목표

- Ravenstack의 사용자 여정을 AARRR 5단계로 구분할 수 있다.
- 각 단계의 분석 목적과 활용 가능한 데이터를 연결할 수 있다.
- 이벤트 로그의 사용자·행동·시점·속성 컬럼을 구분할 수 있다.
- 분석 질문에 맞게 퍼널·코호트·세그멘테이션 분석을 선택할 수 있다.
- 그룹별 Activation 지표를 비교하여 점검 대상을 찾을 수 있다.

## 실습 환경 / 데이터

- Python
- pandas
- `ravenstack_accounts.csv`
- `ravenstack_subscriptions.csv`
- `ravenstack_feature_usage.csv`

Ravenstack은 기업 고객에게 구독형 소프트웨어를 제공하는 B2B SaaS 서비스입니다.

Ravenstack의 AARRR 단계는 다음과 같이 연결할 수 있습니다.

| 단계 | Ravenstack에서의 의미 | 활용 데이터 예시 |
|---|---|---|
| Acquisition | 고객사가 유입되어 가입함 | 가입일, 유입 경로 |
| Activation | 구독 후 제품 기능을 처음 사용함 | 기능 사용 기록 |
| Retention | 제품 사용과 구독을 계속 유지함 | 반복 사용 기록, 이탈 여부 |
| Referral | 기존 고객이 다른 고객을 추천함 | 추천 전송·추천 가입 이벤트 |
| Revenue | 유료 구독에서 반복 매출이 발생함 | MRR, 요금제 |

> 제공된 데이터에는 고객 추천 행동을 직접 기록한 이벤트가 없습니다. `referral_source`는 고객의 유입 경로이며, 기존 고객이 추천 행동을 했다는 사실을 직접 보여주는 Referral 이벤트와는 구분해야 합니다.

## 실습 준비

1. 필요한 라이브러리를 불러오세요.
2. 세 CSV 파일을 각각 `accounts`, `subscriptions`, `feature_usage`에 불러오세요.
3. 각 데이터의 행과 열 개수, 상위 5개 행을 확인하세요.
4. 분석 대상 컬럼의 자료형과 결측치 개수를 확인하세요.
5. `signup_date`, `start_date`, `end_date`, `usage_date`를 날짜형으로 변환하세요.
6. `feature_usage`에서 이벤트 식별자인 `usage_id`의 중복 개수를 확인하세요.
7. 기능별 이벤트 기록 수와 `usage_count`의 기초 통계량을 확인하세요.

In [1]:
# 실습 준비 코드를 작성하세요.

# ── 1. 라이브러리 불러오기 ──
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

FEATURE_COL = "feature_name"   # 기능 컬럼명 (feature_key 등이면 수정)

# ── 2. 세 CSV 불러오기 ──
accounts      = pd.read_csv("ravenstack_accounts.csv",      encoding="utf-8-sig")
subscriptions = pd.read_csv("ravenstack_subscriptions.csv", encoding="utf-8-sig")
feature_usage = pd.read_csv("ravenstack_feature_usage.csv", encoding="utf-8-sig")

dfs = {"accounts": accounts, "subscriptions": subscriptions, "feature_usage": feature_usage}

# ── 3. 행·열 개수와 상위 5개 행 ──
for name, df in dfs.items():
    print(f"===== {name} =====")
    print(f"행 {df.shape[0]:,}개 / 열 {df.shape[1]}개")
    print(f"컬럼: {list(df.columns)}")
    print(df.head())
    print()

# ── 4. 자료형과 결측치 개수 ──
for name, df in dfs.items():
    info = pd.DataFrame({
        "dtype":      df.dtypes,
        "결측치":     df.isna().sum(),
        "결측률(%)":  (df.isna().mean() * 100).round(2),
        "고유값":     df.nunique(),
    })
    print(f"===== {name}: 자료형 · 결측치 =====")
    print(info, "\n")

# ── 5. 날짜형 변환 ──
# errors="coerce": 형식이 깨진 값은 NaT로 → 변환 실패 건수를 셀 수 있음
date_map = {
    "accounts":      ["signup_date"],
    "subscriptions": ["start_date", "end_date"],
    "feature_usage": ["usage_date"],
}
for name, cols in date_map.items():
    df = dfs[name]
    for col in cols:
        before = df[col].isna().sum()
        df[col] = pd.to_datetime(df[col], errors="coerce")
        failed = df[col].isna().sum() - before
        print(f"{name}.{col} → {df[col].dtype}, 변환 실패 {failed}건, "
              f"범위 {df[col].min()} ~ {df[col].max()}")

print(f"\nend_date 결측(진행 중 구독 추정): "
      f"{subscriptions['end_date'].isna().sum():,}건\n")

# ── 6. usage_id 중복 확인 ──
dup_count = feature_usage["usage_id"].duplicated().sum()
print("===== usage_id 중복 점검 =====")
print(f"전체 행       : {len(feature_usage):,}")
print(f"고유 usage_id : {feature_usage['usage_id'].nunique():,}")
print(f"중복 행 수    : {dup_count:,}")
print(f"결측 usage_id : {feature_usage['usage_id'].isna().sum():,}")

if dup_count > 0:
    dup_ids = feature_usage.loc[feature_usage["usage_id"].duplicated(keep=False), "usage_id"]
    print(f"중복된 usage_id 종류: {dup_ids.nunique():,}개")
    print("\n중복 예시:")
    print(feature_usage[feature_usage["usage_id"].isin(dup_ids.unique()[:2])]
          .sort_values("usage_id").to_string(index=False))
    # 필요 시 제거: feature_usage = feature_usage.drop_duplicates("usage_id", keep="first")
print()

# ── 7. 기능별 이벤트 수 & usage_count 기초 통계량 ──
print("===== 기능별 이벤트 기록 수 =====")
feat = pd.DataFrame({
    "이벤트 수": feature_usage[FEATURE_COL].value_counts(dropna=False),
    "비율(%)":   (feature_usage[FEATURE_COL].value_counts(dropna=False, normalize=True) * 100).round(2),
})
print(feat.to_string(), "\n")

print("===== usage_count 기초 통계량 =====")
print(feature_usage["usage_count"].describe())
print(f"중앙값 : {feature_usage['usage_count'].median():,.2f}")
print(f"결측치 : {feature_usage['usage_count'].isna().sum():,}건")
print(f"0 이하 : {(feature_usage['usage_count'] <= 0).sum():,}건\n")

print("===== 기능별 usage_count 요약 =====")
print(feature_usage.groupby(FEATURE_COL)["usage_count"]
      .agg(이벤트수="count", 합계="sum", 평균="mean", 중앙값="median", 최대="max")
      .round(2).sort_values("합계", ascending=False).to_string())


===== accounts =====
행 500개 / 열 10개
컬럼: ['account_id', 'account_name', 'industry', 'country', 'signup_date', 'referral_source', 'plan_tier', 'seats', 'is_trial', 'churn_flag']
  account_id account_name    industry country signup_date referral_source   plan_tier  seats  is_trial  churn_flag
0   A-2e4581    Company_0      EdTech      US  2024-10-16         partner       Basic      9     False       False
1   A-43a9e3    Company_1     FinTech      IN  2023-08-17           other       Basic     18     False        True
2   A-0a282f    Company_2    DevTools      US  2024-08-27         organic       Basic      1     False       False
3   A-1f0ac7    Company_3  HealthTech      UK  2023-08-27           other       Basic     24      True       False
4   A-ce550d    Company_4  HealthTech      US  2024-10-27           event  Enterprise     35     False        True

===== subscriptions =====
행 5,000개 / 열 14개
컬럼: ['subscription_id', 'account_id', 'start_date', 'end_date', 'plan_tier', 'seats', 'mrr

---

## 필수 1. Ravenstack의 AARRR 지표 지도 만들기

### 문제 1-1. 데이터에서 확인 가능한 단계별 대표 지표 계산하기

#### 문제 설명

Ravenstack의 세 테이블을 이용하여 AARRR 단계별 대표 지표를 계산하고, 제공된 데이터만으로 직접 측정하기 어려운 단계도 확인하세요.

이번 문제에서는 다음 기준을 사용합니다.

- Acquisition: 가입한 고유 고객사 수
- Activation: 전체 구독 중 기능 사용 기록이 한 번 이상 있는 구독의 비율
- Retention: 전체 구독 중 현재 `churn_flag`가 `False`인 구독의 비율
- Referral: 제공된 데이터로 직접 계산할 수 없음
- Revenue: 종료일이 없고 체험 구독이 아닌 현재 활성 유료 구독의 MRR 합계

> 여기에서 Retention은 학습을 위한 간단한 대리 지표입니다. 특정 기간 후 다시 사용한 비율을 계산한 정식 리텐션과는 다릅니다.

#### 요구사항

1. 가입한 고유 고객사 수를 `acquired_accounts`로 계산하세요.
2. 기능 사용 기록이 있는 고유 구독 수를 전체 구독 수로 나누어 `activation_rate`를 계산하세요.
3. `churn_flag`가 `False`인 구독의 비율을 `non_churned_rate`로 계산하세요.
4. 종료일이 없고 체험 구독이 아닌 구독의 MRR 합계를 `current_paid_mrr`로 계산하세요.
5. 네 지표를 알아보기 쉬운 형식으로 출력하세요.
6. 각 지표를 AARRR 단계와 연결하고 분석 목적을 설명하세요.
7. Referral 단계를 직접 측정하려면 어떤 이벤트가 추가로 필요한지 제안하세요.

#### 해석 질문

**Q1.** 기능 사용 기록이 있는 구독 비율은 어느 AARRR 단계와 연결되나요?  
**Q2.** `non_churned_rate`를 정식 리텐션율과 동일하게 보면 안 되는 이유는 무엇인가요?  
**Q3.** `referral_source`만으로 Referral 행동을 직접 측정할 수 있나요?  
**Q4.** MRR은 어느 AARRR 단계와 연결되나요?

#### 제출 결과

- AARRR 단계별 대표 지표 계산 코드와 결과
- 각 지표의 단계 및 분석 목적
- Referral 측정에 필요한 이벤트 제안
- Q1~Q4 답변

In [3]:
# 필수 1 코드를 작성하세요.

import pandas as pd
import numpy as np

# ── 1. Acquisition: 가입한 고유 고객사 수 ──
acquired_accounts = accounts["account_id"].nunique()

# ── 2. Activation: 기능 사용 기록이 있는 구독 비율 ──
# feature_usage에 subscription_id가 있다고 가정 (없으면 아래 주석 참고)
used_subs = feature_usage["subscription_id"].dropna().nunique()
total_subs = subscriptions["subscription_id"].nunique()
activation_rate = used_subs / total_subs

# feature_usage에 account_id만 있는 경우:
# used_subs = (subscriptions.merge(
#                  feature_usage[["account_id"]].drop_duplicates(),
#                  on="account_id", how="inner")["subscription_id"].nunique())

# ── 3. Retention: churn_flag가 False인 구독 비율 ──
churn = subscriptions["churn_flag"].astype(bool)
non_churned_rate = (~churn).mean()

# ── 4. Revenue: 종료일 없고 체험이 아닌 구독의 MRR 합계 ──
paid_mask = subscriptions["end_date"].isna() & (~subscriptions["is_trial"].astype(bool))
current_paid_mrr = subscriptions.loc[paid_mask, "mrr_amount"].sum()

# ── 5. 지표 출력 ──
print("=" * 52)
print(f"{'AARRR 핵심 지표':^46}")
print("=" * 52)
print(f"  Acquisition | 가입 고객사 수     : {acquired_accounts:>12,}개")
print(f"  Activation  | 기능 사용 구독 비율: {activation_rate:>12.2%}")
print(f"              |   └ {used_subs:,} / {total_subs:,} 구독")
print(f"  Retention   | 미이탈 구독 비율   : {non_churned_rate:>12.2%}")
print(f"              |   └ 이탈 {int(churn.sum()):,}건")
print(f"  Revenue     | 현재 유료 MRR      : {current_paid_mrr:>12,.0f}")
print(f"              |   └ 활성 유료 {paid_mask.sum():,}건, "
      f"ARR {current_paid_mrr*12:,.0f}")
print("=" * 52)

# 데이터프레임 형태로도 정리
metrics = pd.DataFrame([
    ["Acquisition", "acquired_accounts",   f"{acquired_accounts:,}개"],
    ["Activation",  "activation_rate",     f"{activation_rate:.2%}"],
    ["Retention",   "non_churned_rate",    f"{non_churned_rate:.2%}"],
    ["Revenue",     "current_paid_mrr",    f"{current_paid_mrr:,.0f}"],
    ["Referral",    "(측정 불가)",          "해당 이벤트 없음"],
], columns=["AARRR 단계", "지표명", "값"])
print("\n" + metrics.to_string(index=False))

                 AARRR 핵심 지표                  
  Acquisition | 가입 고객사 수     :          500개
  Activation  | 기능 사용 구독 비율:       99.34%
              |   └ 4,967 / 5,000 구독
  Retention   | 미이탈 구독 비율   :       90.28%
              |   └ 이탈 486건
  Revenue     | 현재 유료 MRR      :   10,159,608
              |   └ 활성 유료 3,814건, ARR 121,915,296

   AARRR 단계               지표명          값
Acquisition acquired_accounts       500개
 Activation   activation_rate     99.34%
  Retention  non_churned_rate     90.28%
    Revenue  current_paid_mrr 10,159,608
   Referral           (측정 불가)  해당 이벤트 없음


### 필수 1 답변 작성란

- **Q1.** -> Activation 단계, 구독한 고객이 제품의 기능을 실제로 사용하여 핵심 가치를 경험했는지를 확인하기 때문
- **Q2.** -> 비이탈 상태만 나타나고, 동일한 사용자 그룹이 가입 후 언제 사용했고 다시 사용했는 지를 시간에 따라 추적하지 않아서
- **Q3.** -> 어렵다. 고객이 어떤 경로로 들어왔는 지 보여주지만, referral_source는 고객이 어떤 경로로 들어왔는지 보여주지만,
이 고객이 다른 고객을 추천해주거나 유도했다는 행동 자체는 보여주지 않는다.
-> 이벤트성 변수가 추가적으로 필요하다.
- **Q4.** -> 

---

## 필수 2. 요금제별 Activation 지표 비교하기

### 문제 2-1. 2024년 12월 제품 활성률을 요금제별로 비교하기

#### 문제 설명

프로덕트팀은 다음 질문에 답하려고 합니다.

> 2024년 12월에 어떤 요금제의 제품 활성률이 상대적으로 낮은가?

이 질문은 사용자를 요금제별로 나누어 비교하므로 **세그멘테이션 분석**에 해당하며, AARRR 중 Activation 단계를 살펴봅니다.

이번 실습에서 제품 활성률은 다음과 같이 정의합니다.

> 12월 제품 활성 유료 구독 수 ÷ 12월 이용 가능 유료 구독 수

#### 요구사항

1. 2024년 12월의 시작일과 마지막 날을 설정하세요.
2. 12월에 이용 가능한 유료 구독을 `eligible_dec`에 저장하세요.
3. 12월에 기능을 사용한 구독 ID를 `active_subscription_ids`로 준비하세요.
4. `eligible_dec`에 `is_product_active` 컬럼을 만드세요.
5. `plan_tier`별로 다음 값을 집계하여 `plan_activation`을 만드세요.
   - 이용 가능 유료 구독 수
   - 제품 활성 유료 구독 수
   - 제품 활성률
6. 제품 활성률이 낮은 순서로 정렬하고 백분율로 출력하세요.
7. 가장 낮은 요금제를 찾아 개선을 위해 추가로 확인할 내용을 제안하세요.
8. 이 질문에 세그멘테이션 분석이 적절한 이유를 설명하세요.

#### 해석 질문

**Q1.** 이 분석은 AARRR 중 어느 단계에 해당하나요?  
**Q2.** 어떤 분석 기법을 사용했으며, 그 이유는 무엇인가요?  
**Q3.** 제품 활성률이 가장 낮은 요금제는 무엇인가요?  
**Q4.** 해당 요금제에서 추가로 어떤 데이터를 확인하면 좋을까요?

#### 제출 결과

- `plan_activation` 집계 코드와 결과
- AARRR 단계와 분석 기법
- 우선 점검할 요금제
- 추가 확인 데이터
- Q1~Q4 답변

In [4]:
# 필수 2 코드를 작성하세요.

import pandas as pd
import numpy as np

# ── 1. 2024년 12월 기간 설정 ──
dec_start = pd.Timestamp("2024-12-01")
dec_end   = pd.Timestamp("2024-12-31")
print(f"분석 기간: {dec_start:%Y-%m-%d} ~ {dec_end:%Y-%m-%d}\n")

# ── 2. 12월에 이용 가능한 유료 구독 ──
# 이용 가능 = 12월 시작 전까지 개시했고(start_date <= 12/31),
#             12월 중 유효함(end_date 결측 또는 12/1 이후 종료), 체험판 아님
started   = subscriptions["start_date"] <= dec_end
not_ended = subscriptions["end_date"].isna() | (subscriptions["end_date"] >= dec_start)
not_trial = ~subscriptions["is_trial"].astype(bool)

eligible_dec = subscriptions[started & not_ended & not_trial].copy()
print(f"12월 이용 가능 유료 구독: {len(eligible_dec):,}건 "
      f"/ 전체 {len(subscriptions):,}건\n")

# ── 3. 12월에 기능을 사용한 구독 ID ──
dec_usage = feature_usage[
    feature_usage["usage_date"].between(dec_start, dec_end)
]
active_subscription_ids = set(dec_usage["subscription_id"].dropna().unique())
print(f"12월 사용 이벤트: {len(dec_usage):,}건")
print(f"12월 사용 구독 수: {len(active_subscription_ids):,}건\n")

# ── 4. is_product_active 컬럼 ──
eligible_dec["is_product_active"] = (
    eligible_dec["subscription_id"].isin(active_subscription_ids)
)
print(f"제품 활성 구독: {eligible_dec['is_product_active'].sum():,}건 "
      f"(전체 활성률 {eligible_dec['is_product_active'].mean():.2%})\n")

# ── 5. plan_tier별 집계 ──
plan_activation = eligible_dec.groupby("plan_tier").agg(
    eligible_subs=("subscription_id", "nunique"),
    active_subs=("is_product_active", "sum"),
)
plan_activation["activation_rate"] = (
    plan_activation["active_subs"] / plan_activation["eligible_subs"]
)

# ── 6. 활성률 오름차순 정렬 + 백분율 출력 ──
plan_activation = plan_activation.sort_values("activation_rate")

print("===== 요금제별 제품 활성률 (낮은 순) =====")
print(pd.DataFrame({
    "이용 가능 유료 구독": plan_activation["eligible_subs"].map("{:,}".format),
    "제품 활성 구독":     plan_activation["active_subs"].astype(int).map("{:,}".format),
    "제품 활성률":        plan_activation["activation_rate"].map("{:.2%}".format),
}).to_string())

overall = eligible_dec["is_product_active"].mean()
print(f"\n전체 평균 활성률: {overall:.2%}")

# ── 7. 활성률 최저 요금제 ──
worst = plan_activation["activation_rate"].idxmin()
w = plan_activation.loc[worst]
print(f"\n▶ 활성률 최저 요금제: {worst}")
print(f"   활성률 {w['activation_rate']:.2%} (평균 대비 {w['activation_rate']-overall:+.2%}p)")
print(f"   {int(w['active_subs']):,} / {int(w['eligible_subs']):,}건")

inactive_mrr = eligible_dec.loc[
    (eligible_dec["plan_tier"] == worst) & (~eligible_dec["is_product_active"]),
    "mrr_amount"
].sum()
print(f"   미사용 구독의 MRR 합계: {inactive_mrr:,.0f} (이탈 위험 노출 매출)")

분석 기간: 2024-12-01 ~ 2024-12-31

12월 이용 가능 유료 구독: 3,981건 / 전체 5,000건

12월 사용 이벤트: 1,076건
12월 사용 구독 수: 973건

제품 활성 구독: 788건 (전체 활성률 19.79%)

===== 요금제별 제품 활성률 (낮은 순) =====
           이용 가능 유료 구독 제품 활성 구독  제품 활성률
plan_tier                              
Pro              1,342      260  19.37%
Basic            1,267      247  19.49%
Enterprise       1,372      281  20.48%

전체 평균 활성률: 19.79%

▶ 활성률 최저 요금제: Pro
   활성률 19.37% (평균 대비 -0.42%p)
   260 / 1,342건
   미사용 구독의 MRR 합계: 1,621,851 (이탈 위험 노출 매출)


### 필수 2 답변 작성란

- **Q1.** -> activation 단계, 유료 구독자가 12월에 제품을 실제로 사용했는지를 확인하려고 했다.
- **Q2.** -> 세그맨테이션 분석을 사용, 요금제별ㄹ로 구독을 나누어 제품의 활성률 차이를 비교했다.
- **Q3.** -> pro, 19.37%로 가장 낮은 제품활성률을 보였다.
- **Q4.** -> 기능별 사용률, 오류 발생률, 기능 사용 시간, 고객 문의 등 추가 데이터 검토 필요

---

## 과제 1. 평가 문항 기반 독립 과제

### 문제 3-1. 결제 주기별 Activation 지표 비교하기

#### 문제 설명

프로덕트팀은 2024년 12월 제품 활성률이 월간 결제 구독과 연간 결제 구독에서 다르게 나타나는지 확인하려고 합니다.

> 이 과제는 필수 2의 요금제별 세그멘테이션을 `billing_frequency`에 동일하게 적용하는 문제입니다.

#### 요구사항

1. 필수 2에서 만든 `eligible_dec`과 `is_product_active`를 사용하세요.
2. `billing_frequency`별로 다음 값을 집계하여 `billing_activation`을 만드세요.
   - 이용 가능 유료 구독 수
   - 제품 활성 유료 구독 수
   - 제품 활성률
3. 제품 활성률이 낮은 순서로 정렬하고 백분율로 출력하세요.
4. 제품 활성률이 더 낮은 결제 주기를 찾으세요.
5. 해당 그룹을 우선 점검 대상으로 정하고 추가로 확인할 내용을 한 가지 제안하세요.
6. 이 분석의 AARRR 단계와 분석 기법을 설명하세요.

#### 해석 질문
**Q1.** 제품 활성률이 더 낮은 결제 주기는 무엇인가요?  
**Q2.** 이 분석은 어떤 AARRR 단계와 분석 기법에 해당하나요?  
**Q3.** 두 그룹의 제품 활성률 차이만으로 결제 주기가 낮은 활성의 원인이라고 결론 내릴 수 있나요?

**Q4.** 퍼널 분석·코호트 리텐션 분석·세그멘테이션을 표로 정리하세요. 각 분석의 계산 또는 분류 원리, 해결하려는 분석 질문, 관련 AARRR 단계를 설명하세요. 전환율·이탈률의 분모, 코호트의 시작 기준과 경과 기간, 세그먼트 분류 기준을 포함하세요.


#### 제출 결과

- `billing_activation` 집계 코드와 결과
- 우선 점검할 결제 주기
- 추가 확인 내용
- AARRR 단계와 분석 기법
- Q1~Q4 답변

In [2]:
# 과제 1 코드를 작성하세요.

import pandas as pd
import numpy as np

# ── 1~2. billing_frequency별 집계 ──
billing_activation = eligible_dec.groupby("billing_frequency").agg(
    eligible_subs=("subscription_id", "nunique"),
    active_subs=("is_product_active", "sum"),
)
billing_activation["activation_rate"] = (
    billing_activation["active_subs"] / billing_activation["eligible_subs"]
)

# ── 3. 활성률 오름차순 정렬 + 백분율 출력 ──
billing_activation = billing_activation.sort_values("activation_rate")

print("===== 결제 주기별 제품 활성률 (낮은 순) =====")
print(pd.DataFrame({
    "이용 가능 유료 구독": billing_activation["eligible_subs"].map("{:,}".format),
    "제품 활성 구독":     billing_activation["active_subs"].astype(int).map("{:,}".format),
    "제품 활성률":        billing_activation["activation_rate"].map("{:.2%}".format),
}).to_string())

overall = eligible_dec["is_product_active"].mean()
print(f"\n전체 평균 활성률: {overall:.2%}")

# ── 4. 활성률이 더 낮은 결제 주기 ──
worst = billing_activation["activation_rate"].idxmin()
w = billing_activation.loc[worst]
best = billing_activation["activation_rate"].idxmax()
b = billing_activation.loc[best]

print(f"\n▶ 활성률 최저 결제 주기: {worst}")
print(f"   {w['activation_rate']:.2%}  ({int(w['active_subs']):,}/{int(w['eligible_subs']):,}건)")
print(f"▶ 비교 대상: {best} — {b['activation_rate']:.2%}")
print(f"   격차: {(b['activation_rate'] - w['activation_rate'])*100:.2f}%p")

# ── 5. 후속 확인용 보조 지표 ──
inactive_mrr = eligible_dec.loc[
    (eligible_dec["billing_frequency"] == worst) & (~eligible_dec["is_product_active"]),
    "mrr_amount"
].sum()
print(f"\n{worst} 미사용 구독의 MRR 합계: {inactive_mrr:,.0f}")

# 교차표: 결제 주기 × 요금제 (혼재 효과 확인)
print("\n===== 결제 주기 × 요금제 활성률 교차표 =====")
cross = eligible_dec.pivot_table(
    index="billing_frequency", columns="plan_tier",
    values="is_product_active", aggfunc="mean"
)
print(cross.map(lambda v: f"{v:.1%}" if pd.notna(v) else "-").to_string())

print("\n===== 결제 주기 × 요금제 구독 수 =====")
print(pd.crosstab(eligible_dec["billing_frequency"], eligible_dec["plan_tier"]).to_string())


NameError: name 'eligible_dec' is not defined

### 과제 1 답변 작성란

- **Q1.** 제품 활성률이 더 낮은 결제 주기는 무엇인가요?
-> 제품 활성률이 더 낮은 결제 주기는 annual이다.

- **Q2.** 이 분석은 어떤 AARRR 단계와 분석 기법에 해당하나요?
-> 고객이 제품을 실제로 사용하며 가치를 느끼는지 확인했으므로 AARRR 중 활성화 단계에 해당한다. 분석 기법은 세그멘테이션이다.

- **Q3.** 두 그룹의 제품 활성률 차이만으로 결제 주기가 낮은 활성의 원인이라고 결론 내릴 수 있나요?
-> 결론 내릴 수 없다. 격차가 0.46%로 작아 우연에 의한 차이일 수 있다. 

- **Q4.** 퍼널 분석·코호트 리텐션 분석·세그멘테이션을 표로 정리하세요. 각 분석의 계산 또는 분류 원리, 해결하려는 분석 질문, 관련 AARRR 단계를 설명하세요. 전환율·이탈률의 분모, 코호트의 시작 기준과 경과 기간, 세그먼트 분류 기준을 포함하세요.

| 구분 | 퍼널 분석 | 코호트 리텐션 분석 | 세그멘테이션 |
|---|---|---|---|
| 원리 | 단계별 전환율을 구함 | 같은 시점 시작 고객을 추적 | 기준별로 나눠 비교 |
| 분모·기준 | 이전 단계 도달자 수 | 코호트 시작 인원 | 각 그룹 대상 수 |

---

## 실습 마무리

아래 질문에 답하세요.

1. Ravenstack의 사용자 여정을 어떤 AARRR 단계로 나누었나요? 
-> 고객사 가입은 Acquisition, 첫 기능 사용은 Activation, 반복 사용과 구독 유지는 Retention

2. 각 단계는 어떤 분석 질문에 답하기 위해 사용했나요?
-> 어디서 고객을 확보했는지, 핵심 기능을 사용했는지?, 계속 사용하는지?, 다른 고객에게도 추천했는지?
-> 실제 매출이 발생했는지 확인

3. 제공된 데이터에서 직접 측정하기 어려운 단계는 무엇이었나요?
-> Referral 단계, 추천이라는 컬럼이 존재하지 않음

4. 요금제와 결제 주기별 비교에는 어떤 분석 기법을 사용했나요?
-> 사용자 특성에 따라 그룹을 나누어 비교하는 세그멘테이션 분석 사용

5. 분석 결과를 근거로 어떤 대상을 우선 점검했나요?
-> 요금제에서는 Pro 결제주기에서는 annual 그룹의 제품 활성률이 낮아 우선 점검대상으로 정했다.